In [ ]:
!pip install "protobuf<=3.20.3"

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import cv2
import random
import pandas as pd
import numpy as np
import shutil
import math
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from keras import layers, models
import keras.backend as K
from keras.optimizers import Adam, RMSprop
from keras.layers import Input, Concatenate, ZeroPadding2D, BatchNormalization
from keras.layers import Dense, Dropout, Activation
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from keras.models import Model, load_model
from keras.models import Sequential
from keras.preprocessing import image
from keras.applications import DenseNet121
from keras.applications.densenet import preprocess_input

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from keras.callbacks import ModelCheckpoint
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("TF version:", tf.version)
print("Keras version:", tf.keras.version)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


#!pip install pyyaml

In [ ]:
data_path = (
    "/kaggle/input/datasets/dewamardana/dataset-manual-selection/dataset_centralcrop"
)

images, labels = []

for subfolder in os.listdir(data_path):

    subfolder_path = os.path.join(data_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for image_filename in os.listdir(subfolder_path):
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)

        labels.append(subfolder)

data = pd.DataFrame({"image": images, "label": labels})
data.head()
data.shape

In [ ]:
strat = data["label"]
train_df, dummy_df = train_test_split(
    data, train_size=0.80, shuffle=True, random_state=123, stratify=strat
)

strat = dummy_df["label"]
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True, random_state=123, stratify=strat
)

In [ ]:
print("Training set shape:", train_df.shape)
print("Validation set shape:", valid_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
batch_size = 4
img_size = (256, 256)
channels = 3
img_shape = (img_size[0], img_size[1], channels)


tr_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

ts_gen = ImageDataGenerator()

feat_gen = ImageDataGenerator()

train_gen_noaug = feat_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="sparse",
    shuffle=False,
    batch_size=batch_size,
)

train_gen = tr_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    batch_size=batch_size,
)

valid_gen = ts_gen.flow_from_dataframe(
    valid_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

test_gen = ts_gen.flow_from_dataframe(
    test_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

In [ ]:
num_classes = 12
epochs = 16

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,  # Tanpa layer klasifikasi asli
    pooling="avg",  # Global average pooling sebagai output
    input_shape=(256, 256, 3),
)

for layer in base_model.layers:
    layer.trainable = False

unfreeze = False
for layer in base_model.layers:
    if "conv4_block1" in layer.name:  # mulai unfreeze dari block 4
        unfreeze = True
    if unfreeze:
        layer.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

x = layers.Dense(512, activation="relu")(base_model.output)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation="relu", name="feature_output")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)


model = models.Model(inputs=base_model.input, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

filepath = "model_4.keras"
checkpoint = ModelCheckpoint(
    filepath, monitor="val_accuracy", verbose=1, save_best_only=True, mode="max"
)
callbacks_list = [checkpoint]

cnn_training_start = time.time()

history = model.fit(
    train_gen, validation_data=valid_gen, epochs=epochs, callbacks=callbacks_list
)

cnn_training_end = time.time()

cnn_training_time = cnn_training_end - cnn_training_start

print(f"CNN Training Time: " f"{cnn_training_time:.2f} seconds")

model.save("model_4_Final.keras")

In [ ]:
# =============================
# 1. Evaluasi Train & Validation (dari history)
# =============================
train_acc = history.history["accuracy"][-1]
train_loss = history.history["loss"][-1]
val_acc = history.history["val_accuracy"][-1]
val_loss = history.history["val_loss"][-1]

print("=== TRAINING METRICS ===")
print("Training Accuracy :", train_acc)
print("Training Loss     :", train_loss)
print("\n=== VALIDATION METRICS ===")
print("Validation Accuracy :", val_acc)
print("Validation Loss     :", val_loss)

In [ ]:
# =============================
# 2. Plot Training vs Validation Accuracy & Loss
# =============================
import matplotlib.pyplot as plt

epochs = range(len(history.history["accuracy"]))

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Accuracy")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["loss"], label="Training Loss")
plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Loss")
plt.show()

In [ ]:
from tensorflow.keras.utils import to_categorical

# =============================
# 3. Prediksi Validation Set
# =============================
ypred = model.predict(valid_gen, verbose=1)

# ground truth
ytrue = valid_gen.classes
ytrue_cat = to_categorical(ytrue, num_classes=num_classes)

In [ ]:
# =============================
# 4. Evaluasi Test Dataset
# =============================
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print("\n=== TEST METRICS ===")
print("Test Accuracy :", test_acc)
print("Test Loss     :", test_loss)

In [ ]:
# =============================
# 5. Akurasi Manual
# =============================
predicted_labels = np.argmax(ypred, axis=1)
accurate = np.sum(predicted_labels == ytrue)
total = len(ytrue)

print("\n=== MANUAL ACCURACY CHECK ===")
print("Total Data     :", total)
print("Correct Predict:", accurate)
print("Wrong Predict  :", total - accurate)
print("Accuracy (%)   :", accurate / total * 100)

In [ ]:
# =============================
# 6. Log Loss
# =============================
from sklearn.metrics import log_loss

val_logloss = log_loss(ytrue_cat, ypred)
print("\nValidation Log Loss:", val_logloss)

In [ ]:
# =============================
# 7. Confusion Matrix & Classification Report
# =============================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(ytrue, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("\n=== Classification Report ===")
print(
    classification_report(
        ytrue, predicted_labels, target_names=list(train_gen.class_indices.keys())
    )
)

In [ ]:
# =====================================================
# 🔹 LOAD MODEL CNN (DenseNet)
# =====================================================
# if (!model):
# model = load_model("/kaggle/input/models/dewamardana/model-final-skema-4-densenet-121/keras/default/1/model_4_Final.keras")

model.summary(expand_nested=True, line_length=200)

In [ ]:
from sklearn.preprocessing import StandardScaler
import gc

# =====================================================
# ⚙️ KONFIGURASI DASAR
# =====================================================
img_size = (256, 256)
channels = 3

# Ambil output dari layer global average pooling terakhir
feature_model = Model(inputs=model.input, outputs=model.get_layer("avg_pool").output)

# feature_model = Model(
#     inputs=model.input,
#     outputs=model.get_layer("feature_output").output   # atau "avg_pool"
# )


# =====================================================
# 🔍 EKSTRAKSI FITUR
# =====================================================
def extract_features(generator, feature_extractor):
    features = feature_extractor.predict(generator, verbose=1)
    labels = np.array(generator.classes)
    return features, labels


# Ekstraksi fitur
print("Ekstraksi fitur training...")

feature_start_time = time.time()
train_features, train_labels = extract_features(train_gen_noaug, feature_model)
test_features, test_labels = extract_features(test_gen, feature_model)


# # =====================================================
# # 🔹 Encode label ke integer
# # =====================================================
train_labels = train_gen_noaug.classes
test_labels = test_gen.classes
# =====================================================
# ⚖️ NORMALISASI FITUR
# =====================================================
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

feature_end_time = time.time()

feature_extraction_time = feature_end_time - feature_start_time

print(f"Feature Extraction Time: " f"{feature_extraction_time:.2f} seconds")

# =====================================================
# 🔹 HAPUS OBJEK TIDAK DIPAKAI UNTUK HEMAT MEMORI
# =====================================================
gc.collect()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Jika ingin GPU:
# from cuml.svm import SVC             # GPU
# Jika CPU:
from sklearn.svm import SVC  # CPU

# ============================================
# 3. Inisialisasi model SVM
# ============================================
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=False)

print("Training SVM sedang berjalan...\n")

# ============================================
# 4. Training
# ============================================
svm_training_start = time.time()
svm_model.fit(train_features, train_labels)
svm_training_end = time.time()

svm_training_time = svm_training_end - svm_training_start

cnn_training_time = 3517.95
total_training_time = cnn_training_time + feature_extraction_time + svm_training_time

print(f"SVM Training Time: " f"{svm_training_time:.2f} seconds")

print("Training selesai!\n")

In [ ]:
# ==========================================
# CLASSIFICATION START
# ==========================================
print("Ekstraksi fitur testing...")
classification_start_time = time.time()


# SVM prediction
svm_predictions = svm_model.predict(test_features)

# ==========================================
# CLASSIFICATION END
# ==========================================
classification_end_time = time.time()

total_classification_time = classification_end_time - classification_start_time

print(f"Classification Time: " f"{total_classification_time:.2f} seconds")

# Optional probability
# svm_probabilities = (
#     svm_model.predict_proba(test_features)
# )

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(test_labels, svm_predictions)

precision = precision_score(test_labels, svm_predictions, average="weighted")

recall = recall_score(test_labels, svm_predictions, average="weighted")

f1 = f1_score(test_labels, svm_predictions, average="weighted")

# =====================================================
# CLASSIFICATION REPORT
# =====================================================
print("\n📌 Classification Report:")
print(
    classification_report(
        test_labels, svm_predictions, target_names=list(test_gen.class_indices.keys())
    )
)

In [ ]:
import joblib

joblib.dump(svm_model, "svm_model_skema4.pkl")

model.save("cnn_skema4.keras")

cnn_model_size = os.path.getsize("cnn_skema4.keras") / (1024 * 1024)

svm_model_size = os.path.getsize("svm_model_skema4.pkl") / (1024 * 1024)

total_model_size = cnn_model_size + svm_model_size

In [ ]:
print("\n===================================")
print("FINAL RESULT - SKEMA 4")
print("===================================")

print(f"Accuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1-Score               : {f1:.4f}")

print(f"CNN Training Time(s)  : {cnn_training_time:.2f}")
print(f"Feature Extraction(s) : {feature_extraction_time:.2f}")
print(f"SVM Training Time(s)  : {svm_training_time:.2f}")
print(f"Total Training Time(s): {total_training_time:.2f}")

print(f"Classification Time(s): {total_classification_time:.2f}")

print(f"CNN Model Size(MB)    : {cnn_model_size:.2f}")
print(f"SVM Model Size(MB)    : {svm_model_size:.2f}")
print(f"Total Model Size(MB)  : {total_model_size:.2f}")

In [ ]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.svm import SVC
# import numpy as np
# import gc

# # =====================================================
# # ⚙️ PERSIAPAN K-FOLD
# # =====================================================
# X = data["image"].values   # path gambar
# y = data["label"].values   # label

# k = 5
# skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=123)

# all_scores = []
# fold_no = 1

# # =====================================================
# # 🔄 LOOP K-FOLD
# # =====================================================
# for train_index, test_index in skf.split(X, y):

#     print(f"\n==============================")
#     print(f"        FOLD {fold_no}")
#     print(f"==============================")

#     # Buat dataframe train/test per fold
#     train_df = data.iloc[train_index].reset_index(drop=True)
#     test_df  = data.iloc[test_index].reset_index(drop=True)

#     # -------------------------------------------------
#     # 1. Buat generator untuk fold ini
#     # -------------------------------------------------
#     train_gen_fold = feat_gen.flow_from_dataframe(
#         train_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     test_gen_fold = ts_gen.flow_from_dataframe(
#         test_df,
#         x_col='image',
#         y_col='label',
#         target_size=img_size,
#         class_mode='sparse',
#         shuffle=False,
#         batch_size=batch_size
#     )

#     # -------------------------------------------------
#     # 2. Ekstraksi fitur CNN untuk fold ini
#     # -------------------------------------------------
#     print("Ekstraksi fitur training...")
#     train_features, train_labels = extract_features(train_gen_fold, feature_model)

#     print("Ekstraksi fitur test...")
#     test_features, test_labels = extract_features(test_gen_fold, feature_model)

#     # -------------------------------------------------
#     # 3. Normalisasi
#     # -------------------------------------------------
#     scaler = StandardScaler()
#     train_features = scaler.fit_transform(train_features)
#     test_features  = scaler.transform(test_features)

#     # -------------------------------------------------
#     # 4. Train SVM
#     # -------------------------------------------------
#     svm = SVC(kernel="rbf", C=1.0, gamma="scale")
#     svm.fit(train_features, train_labels)

#     # -------------------------------------------------
#     # 5. Evaluasi
#     # -------------------------------------------------
#     preds = svm.predict(test_features)

#     acc = accuracy_score(test_labels, preds)
#     all_scores.append(acc)

#     print(f"Accuracy Fold {fold_no}: {acc*100:.2f}%\n")
#     print(classification_report(test_labels, preds))
#     print(confusion_matrix(test_labels, preds))

#     fold_no += 1
#     gc.collect()


# # =====================================================
# # 📊 HASIL FINAL
# # =====================================================
# print("\n================================")
# print("        FINAL K-FOLD RESULT")
# print("================================")
# for i, score in enumerate(all_scores, start=1):
#     print(f"Fold {i}: {score*100:.2f}%")

# print("\nAverage Accuracy:", np.mean(all_scores)*100, "%")

In [ ]:
import os
import json
import numpy as np

# =====================================================
# 3️⃣ SAVE RBF SVM (ANDROID COMPATIBLE)
# =====================================================
print("Saving RBF SVM parameters for Android...")

print("🔍 VALIDATION CHECK")
print("Support vectors:", svm_model.support_vectors_.shape)
print("Dual coef shape :", svm_model.dual_coef_.shape)
print("Intercept shape :", svm_model.intercept_.shape)
print("Gamma used      :", svm_model._gamma)

# -----------------------------------------------------
# 📁 OUTPUT DIRECTORY (KAGGLE)
# -----------------------------------------------------
OUTPUT_DIR = "/kaggle/working/android_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)


svm_data = {
    "kernel": "rbf",
    "gamma": float(svm_model._gamma),  # REAL gamma value
    "n_classes": len(np.unique(train_labels)),
    "support_vectors": svm_model.support_vectors_.tolist(),
    "dual_coef": svm_model.dual_coef_.tolist(),
    "intercept": svm_model.intercept_.tolist(),
}

svm_rbf_path = os.path.join(OUTPUT_DIR, "svm_rbf_12class.json")
with open(svm_rbf_path, "w") as f:
    json.dump(svm_data, f)

print(f"✔ Saved: {svm_rbf_path}")